# 📏 Why Evaluation Matters

**Day 9 — Notebook 1 of 4 | Estimated Time: 20 minutes**

---

## What You'll Learn
- Why "it looks good" is not a valid evaluation strategy
- The difference between offline and online evaluation
- What a golden dataset is and how to build one
- The evaluation mindset: treat AI outputs like software under test

---

## The Evaluation Gap

Most teams building LLM applications follow this pattern:

```
1. Build the system
2. Try a few prompts manually
3. "Looks good!" → ship it
4. Users find problems in production
```

This is equivalent to writing software without tests.  
Evaluation gives you a systematic way to know whether your system is working.

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" chromadb

In [ ]:
import sys, os, json
import chromadb

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from pydantic import BaseModel
from IPython.display import Markdown

from google.genai import errors

try:
    client = genai.Client(api_key=get_api_key())
except errors.APIError as e:
    print(f"Gemini API Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
else:
    MODEL_ID = "gemini-2.5-flash"
    print("✅ Ready!")
GEN_MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL = "gemini-embedding-001"

---

## 1. Offline vs Online Evaluation

There are two fundamental evaluation modes, used at different stages of development:

In [ ]:
print("""
OFFLINE EVALUATION (before shipping)
─────────────────────────────────────
What:   Run your system against a fixed dataset with known correct answers
When:   During development, before each release
How:    Compare outputs to ground truth; compute metrics automatically
Goal:   Catch regressions; measure improvement from changes

Examples:
  • "Does the RAG system return the correct answer for 95% of test questions?"
  • "Did this prompt change improve faithfulness score from 3.2 → 3.7?"
  • "Are reranked results more relevant than dense-only on our golden set?"


ONLINE EVALUATION (in production)
──────────────────────────────────
What:   Measure real-world behaviour with actual users
When:   After shipping; continuously
How:    User feedback, implicit signals (thumbs up/down, click-throughs)
Goal:   Detect drift, discover edge cases, measure business impact

Examples:
  • User upvote/downvote rates on answers
  • A/B test: System A vs System B on 50% of traffic each
  • Tracking "answer accepted" vs "follow-up question asked" rates


Both are necessary — offline prevents regressions; online catches real-world failures.
""")

---

## 2. What Is a Golden Dataset?

A **golden dataset** is a curated set of (input, expected output) pairs  
that represents the full range of tasks your system should handle.

It's called "golden" because it's the ground truth — the answers you know are correct.

In [ ]:
# Example golden dataset for a customer support RAG system
GOLDEN_DATASET = [
    # Format: question, ground_truth_answer, relevant_doc_ids, category
    {
        "id": "q001",
        "question": "What is the return policy for electronics?",
        "ground_truth": "Electronics can be returned within 30 days of purchase with original packaging.",
        "relevant_doc_ids": ["returns-policy"],
        "category": "returns",
    },
    {
        "id": "q002",
        "question": "How do I cancel my subscription?",
        "ground_truth": "Go to Account Settings > Subscriptions > Cancel. Cancellation takes effect at the end of the billing period.",
        "relevant_doc_ids": ["account-management"],
        "category": "account",
    },
    {
        "id": "q003",
        "question": "Do you offer student discounts?",
        "ground_truth": "Yes, 15% off with a valid .edu email address. Verification required annually.",
        "relevant_doc_ids": ["pricing", "discounts"],
        "category": "pricing",
    },
    {
        "id": "q004",
        "question": "What payment methods are accepted?",
        "ground_truth": "Visa, Mastercard, Amex, PayPal, and Apple Pay. Bank transfers for orders over $500.",
        "relevant_doc_ids": ["payments"],
        "category": "payments",
    },
    {
        "id": "q005",
        "question": "How long does standard shipping take?",
        "ground_truth": "Standard shipping takes 5-7 business days within the continental US.",
        "relevant_doc_ids": ["shipping"],
        "category": "shipping",
    },
    {
        "id": "q006",
        "question": "Can I change my order after placing it?",
        "ground_truth": "Orders can be modified within 1 hour of placement. After that, you must wait for delivery and use the returns process.",
        "relevant_doc_ids": ["orders"],
        "category": "orders",
    },
]

print(f"Golden dataset: {len(GOLDEN_DATASET)} examples")
print(f"Categories: {set(q['category'] for q in GOLDEN_DATASET)}")
print()
for q in GOLDEN_DATASET:
    print(f"  [{q['id']}] ({q['category']}) {q['question']}")
    print(f"         ↳ {q['ground_truth'][:70]}")

---

## 3. Building a Golden Dataset with LLM Assistance

Manually writing test cases is slow. Use an LLM to generate question–answer pairs  
from your documents, then review and curate them.

In [ ]:
# Source document to generate questions from
SAMPLE_DOC = {
    "id": "shipping-policy",
    "text": """Shipping Policy

Standard shipping (5-7 business days) is free on orders over $50. Orders under $50 incur 
a $4.99 shipping fee. Express shipping (2 business days) costs $12.99. Overnight shipping 
is available for $24.99 on orders placed before 2pm EST Monday–Friday.

International shipping is available to 45 countries. Rates and delivery times vary by 
destination. Import duties and taxes are the responsibility of the recipient. We ship 
using DHL Express for all international orders with tracking provided.

All orders are processed within 1 business day. You will receive a tracking number by 
email once your order ships. In case of lost packages, please contact support within 
30 days of the expected delivery date.""",
}


class QAPair(BaseModel):
    question: str
    answer: str          # Exact answer from the document
    difficulty: str      # "easy", "medium", or "hard"
    answer_type: str     # "factual", "conditional", "procedural"


class QAList(BaseModel):
    pairs: list[QAPair]


def generate_golden_pairs(doc_text: str, n: int = 5) -> list[QAPair]:
    """Generate Q&A pairs from a document using Gemini."""
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Generate {n} diverse question-answer pairs from the following document.
Requirements:
- Questions should be realistic user questions, not document summaries
- Answers must be grounded in the document text — no inference
- Include a mix of easy (single fact), medium (combining facts), and hard (conditional) questions
- Include factual ("what is"), conditional ("what if"), and procedural ("how do I") types

Document:
{doc_text}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=QAList,
            temperature=0.4,
        ),
    )
    return QAList.model_validate_json(response.text).pairs


print("Generating golden Q&A pairs from shipping policy...")
pairs = generate_golden_pairs(SAMPLE_DOC["text"], n=5)

print(f"\nGenerated {len(pairs)} pairs:\n")
for i, p in enumerate(pairs):
    print(f"  [{i+1}] [{p.difficulty}/{p.answer_type}] Q: {p.question}")
    print(f"       A: {p.answer}")
    print()

---

## 4. Evaluation Dimensions

Not every system needs the same metrics. Here's a map of what to measure:

In [ ]:
print("""
WHAT TO EVALUATE — AND WHERE IT COMES FROM

For any LLM output:
┌────────────────────┬────────────────────────────────────────────┐
│ Dimension          │ Question It Answers                        │
├────────────────────┼────────────────────────────────────────────┤
│ Correctness        │ Is the answer factually right?             │
│ Faithfulness       │ Is the answer grounded in the source?      │
│ Relevance          │ Does it address what was asked?            │
│ Coherence          │ Is it logically consistent and readable?   │
│ Conciseness        │ Is it appropriately brief?                 │
└────────────────────┴────────────────────────────────────────────┘

For RAG retrieval:
┌────────────────────┬────────────────────────────────────────────┐
│ Dimension          │ Question It Answers                        │
├────────────────────┼────────────────────────────────────────────┤
│ Precision@K        │ Of K retrieved docs, how many are relevant?│
│ Recall@K           │ Of all relevant docs, how many did we find?│
│ MRR                │ How high is the first relevant result?     │
│ Context Relevance  │ Is retrieved context useful for the query? │
└────────────────────┴────────────────────────────────────────────┘

Priority depends on use case:
  Customer support bot  → Faithfulness > Correctness > Conciseness
  Legal document Q&A    → Faithfulness > Correctness > Relevance
  Creative writing tool → Coherence > Relevance > Conciseness
  Search / RAG system   → Precision@K + Recall@K + Context Relevance
""")

---

## 5. A Simple Baseline Evaluator

Before building a full pipeline, let's establish the simplest possible evaluator:  
exact string match and keyword overlap.

In [ ]:
def keyword_overlap_score(prediction: str, ground_truth: str) -> float:
    """
    Measures what fraction of key words in ground_truth appear in prediction.
    Simple but surprisingly useful for factual Q&A.
    """
    # Normalise: lowercase, remove punctuation
    import re
    def normalise(text):
        text = text.lower()
        text = re.sub(r"[^a-z0-9\s]", "", text)
        return set(text.split())

    # Common English stop words to exclude
    stopwords = {"the", "a", "an", "is", "are", "was", "were", "be", "been",
                 "have", "has", "do", "does", "did", "will", "would", "can",
                 "could", "should", "may", "might", "to", "of", "in", "on",
                 "at", "for", "with", "by", "from", "and", "or", "but", "not"}

    gt_words = normalise(ground_truth) - stopwords
    pred_words = normalise(prediction) - stopwords

    if not gt_words:
        return 1.0
    return len(gt_words & pred_words) / len(gt_words)


# Test with examples
test_cases = [
    {
        "gt": "Electronics can be returned within 30 days of purchase with original packaging.",
        "good_pred": "You can return electronics within 30 days. The original packaging is required.",
        "bad_pred": "We accept returns. Please contact customer support for assistance.",
    },
    {
        "gt": "Standard shipping takes 5-7 business days within the continental US.",
        "good_pred": "Standard shipping is 5-7 business days in the continental US.",
        "bad_pred": "Shipping typically takes about a week depending on location.",
    },
]

print("Keyword overlap scores:")
print(f"{'Case':<6} {'Good pred':>12} {'Bad pred':>12}")
print("-" * 35)
for i, tc in enumerate(test_cases):
    good_score = keyword_overlap_score(tc["good_pred"], tc["gt"])
    bad_score = keyword_overlap_score(tc["bad_pred"], tc["gt"])
    print(f"  {i+1}       {good_score:.2f}           {bad_score:.2f}")

print("""
💡 Keyword overlap is useful for quick sanity checks but misses:
   - Paraphrasing ("five to seven" ≠ "5-7" by this metric)
   - Hallucinations that include correct keywords by accident
   → Next: LLM-as-judge for nuanced semantic evaluation (Notebook 2)
""")

---

## 6. Evaluation Anti-Patterns

Common mistakes that make evaluations misleading:

In [ ]:
print("""
EVALUATION ANTI-PATTERNS TO AVOID

❌ Testing on examples you tuned the prompt for
   → You'll get inflated scores because the prompt was optimised for these exact cases
   → Fix: Hold out a test set; never use it for prompt development

❌ Too few examples (< 50 for most tasks)
   → A 3-example golden set is nearly useless — variance dominates
   → Fix: Build at least 50 diverse examples; 200+ for production systems

❌ All examples from one category or difficulty level
   → Your system may ace easy questions but fail on edge cases
   → Fix: Stratify by category, difficulty, and edge cases

❌ Evaluating the same model on itself (the judge and the examinee are the same model)
   → The model is biased toward its own style and reasoning
   → Fix: Use a different model as judge, or use human evaluation for critical dimensions

❌ Only measuring average scores
   → A system that gets 90% right but catastrophically fails on 10% is dangerous
   → Fix: Also track worst-case scores, failure modes by category, and hard examples

❌ Not versioning your evaluation dataset
   → If your golden set changes, scores become incomparable across runs
   → Fix: Treat the eval set like code — version-control it
""")

---

## 🏋️ Exercise 1: Generate Your Own Golden Dataset

Pick a document from any domain and generate a curated golden dataset.

In [ ]:
# Exercise 1: Build a domain-specific golden dataset
# TODO:
# 1. Write a document about a topic you know well
#    (company policy, a hobby, a technical specification, etc.)
#
# 2. Use generate_golden_pairs() to generate 8 Q&A pairs
#
# 3. Review each pair and identify:
#    - Which pairs are too easy / too hard?
#    - Are any answers wrong, vague, or not grounded in the doc?
#    - Are there important topics the LLM missed?
#
# 4. Manually edit/add pairs to fix issues. Your final set should:
#    - Have at least 8 pairs
#    - Cover all major topics in the document
#    - Include at least 2 edge-case or conditional questions
#
# 5. Save to a JSON variable called MY_GOLDEN_DATASET

MY_DOCUMENT = """TODO: Replace with your document text"""

# generated = generate_golden_pairs(MY_DOCUMENT, n=8)
# for p in generated:
#     print(f"Q: {p.question}\nA: {p.answer}\n")

MY_GOLDEN_DATASET = [
    # {"id": "q001", "question": "...", "ground_truth": "...", "category": "..."},
]

---

## 🏋️ Exercise 2: Diagnose an Evaluation Problem

Given a "golden dataset" with issues, identify the anti-patterns present.

In [ ]:
# Exercise 2: Spot the anti-patterns
# The following dataset was used to evaluate a chatbot. The team reported 95% accuracy.
# But users are complaining the chatbot gives wrong answers.
#
# Look at the dataset and identify what's wrong.
# Write your findings as comments below.

SUSPICIOUS_DATASET = [
    {"q": "What is your return policy?", "gt": "30 days", "category": "returns"},
    {"q": "Can I return items?", "gt": "Yes, 30 days", "category": "returns"},
    {"q": "How long do I have to return?", "gt": "30 days", "category": "returns"},
    {"q": "Return policy?", "gt": "30 days return window", "category": "returns"},
    {"q": "What's your shipping cost?", "gt": "Free over $50", "category": "shipping"},
    {"q": "Is shipping free?", "gt": "Free over $50", "category": "shipping"},
]

# TODO: List the anti-patterns you observe:
# Anti-pattern 1: ...
# Anti-pattern 2: ...
# Anti-pattern 3: ...

# TODO: Rewrite the dataset to fix the issues (aim for 6 better examples)
IMPROVED_DATASET = [
    # Your improved examples here
]

---

## Key Takeaways

| Concept | Detail |
|---------|--------|
| Offline eval | Fixed dataset, run before each change, catches regressions |
| Online eval | Real users, A/B testing, implicit signals — catches real-world failures |
| Golden dataset | Curated (input, expected output) pairs — ground truth for metrics |
| LLM-assisted generation | Use an LLM to draft Q&A pairs, then review and curate |
| Keyword overlap | Fast baseline metric — good for sanity checks, misses paraphrasing |
| Anti-patterns | Overlap between test/train, too few examples, single-category bias |

---

## 📚 Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [Evals as a First-Class Citizen](https://hamel.dev/blog/posts/evals/) | Blog | Hamel Husain on building eval culture |
| [Your AI Product Needs Evals](https://hamel.dev/blog/posts/evals/) | Blog | Practical guide to building eval datasets |
| [BEIR Benchmark](https://github.com/beir-cellar/beir) | Repo | Heterogeneous retrieval evaluation benchmark |

---

**Next up:** [02_LLM_Output_Evaluation.ipynb](./02_LLM_Output_Evaluation.ipynb)